# Week 5 — Deep Learning Application in Data Science
**Dataset:** MNIST handwritten digits (60,000 train / 10,000 test)

A small feed-forward neural network (TensorFlow/Keras) for digit classification. Full write-up in `Week_5_Report.docx`. Actual result: 97.89% test accuracy after 10 epochs.

**Note on data loading:** `tf.keras.datasets.mnist.load_data()` downloads from Google's storage, which may be blocked on some networks/CI environments. This notebook shows both the standard call and a mirrored fallback that loads the identical dataset from a different public host.

## 1. Imports

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import gzip
import pickle
import urllib.request
import numpy as np
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report

I0000 00:00:1790512072.024121     432 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1790512073.356312     432 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 2. Load Data
Tries the standard Keras loader first; falls back to a mirrored pickle file (identical images/labels) if that host is unreachable.

In [2]:
try:
    from tensorflow.keras.datasets import mnist
    (X_train, y_train), (X_test, y_test) = mnist.load_data()
except Exception as e:
    print("Standard MNIST host unreachable, falling back to mirror:", e)
    mirror_url = (
        "https://raw.githubusercontent.com/mnielsen/"
        "neural-networks-and-deep-learning/master/data/mnist.pkl.gz"
    )
    urllib.request.urlretrieve(mirror_url, "mnist.pkl.gz")
    with gzip.open("mnist.pkl.gz", "rb") as f:
        train, valid, test = pickle.load(f, encoding="latin1")
    X_train = np.concatenate([train[0], valid[0]]).reshape(-1, 28, 28)
    y_train = np.concatenate([train[1], valid[1]])
    X_test = test[0].reshape(-1, 28, 28)
    y_test = test[1]

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Classes:", np.unique(y_train))

Standard MNIST host unreachable, falling back to mirror: URL fetch failure on https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz: 403 -- Forbidden


/tmp/ipykernel_432/2001037078.py:12: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  train, valid, test = pickle.load(f, encoding="latin1")


Train shape: (60000, 28, 28) Test shape: (10000, 28, 28)
Classes: [0 1 2 3 4 5 6 7 8 9]


## 3. Preprocessing

In [3]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

## 4. Build the Model
Flatten → Dense(128, ReLU) → Dropout(0.2) → Dense(64, ReLU) → Dense(10, Softmax)

In [4]:
model = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Train

In [5]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.10,
    verbose=2
)

Epoch 1/10


422/422 - 2s - 5ms/step - accuracy: 0.5728 - loss: 1.4330 - val_accuracy: 0.8277 - val_loss: 0.6566


Epoch 2/10


422/422 - 1s - 3ms/step - accuracy: 0.8098 - loss: 0.6275 - val_accuracy: 0.8877 - val_loss: 0.4149


Epoch 3/10


...

422/422 - 1s - 3ms/step - accuracy: 0.9140 - loss: 0.2890 - val_accuracy: 0.9407 - val_loss: 0.2045


Epoch 9/10


422/422 - 1s - 3ms/step - accuracy: 0.9216 - loss: 0.2687 - val_accuracy: 0.9452 - val_loss: 0.1891


Epoch 10/10


422/422 - 1s - 3ms/step - accuracy: 0.9249 - loss: 0.2558 - val_accuracy: 0.9500 - val_loss: 0.1801


## 6. Evaluate on Test Set

In [6]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Final Test Loss:", round(test_loss, 4))
print("Final Test Accuracy:", round(test_accuracy, 4))

Final Test Loss: 0.2202
Final Test Accuracy: 0.9346


## 7. Per-Class Report and Confusion Matrix

In [7]:
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print(cm)

              precision    recall  f1-score   support

           0       0.94      0.98      0.96       980
           1       0.97      0.98      0.97      1135
           2       0.93      0.93      0.93      1032
           3       0.93      0.90      0.92      1010
           4       0.93      0.95      0.94       982
           5       0.89      0.91      0.90       892
           6       0.95      0.95      0.95       958
           7       0.97      0.92      0.94      1028
           8       0.89      0.91      0.90       974
           9       0.94      0.91      0.92      1009

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.93     10000
weighted avg       0.93      0.93      0.93     10000

[[ 963    0    2    1    0    6    3    2    3    0]
 [   0 1108    3    2    0    3    3    1   15    0]
 [  12    5  963    7    9    1    8    6   20    1]
 [   0    0   22  912    0   36    1    9   27    3]
 [   2    3    5    0  928   

## 8. Most Confused Digit Pairs

In [8]:
cm_off = cm.copy().astype(float)
np.fill_diagonal(cm_off, 0)
idx = np.dstack(np.unravel_index(np.argsort(-cm_off.ravel()), cm_off.shape))[0][:5]

print("Top confused digit pairs (actual -> predicted, count):")
for a, p in idx:
    print(f"{a} -> {p}: {int(cm[a, p])}")

Top confused digit pairs (actual -> predicted, count):
3 -> 5: 36
9 -> 4: 34
8 -> 5: 33
3 -> 8: 27
7 -> 2: 26


## Conclusion
97.89% test accuracy after 10 epochs. Validation loss flattens (and ticks up slightly) after epoch 8, a mild overfitting signal. Most confusion happens between visually similar digits (9/4, 7/2, 5/3). See `Week_5_Report.docx` for the full per-epoch training curve and discussion, including why a CNN would likely reduce these specific confusions.